# AES Renewable Asset Performance Triage — Proof of Concept

**Solar-only POC.** Demonstrates the full analytical core end-to-end:

`raw signals → expected baseline (Performance Ratio) → anomaly band → rule-based cause attribution → financial ranking → daily triage table`

The baseline is a **Performance-Ratio (PR) decomposition**:

$$\text{expected\_generation} = \underbrace{\text{capacity}\cdot\frac{\text{POA}}{1000}\cdot(1-\gamma(T_{cell}-25))}_{\text{theoretical potential (physics)}} \times \underbrace{PR_{baseline}}_{\text{learned from healthy history}}$$

The physics term needs no training data; `PR_baseline` (the asset's own healthy median PR) empirically absorbs wiring losses, inverter efficiency, soiling and install geometry we don't have documented. The residual is dimensionless (PR), so deviations are **comparable across a 5 MW and a 200 MW plant**.

In [1]:
%pip install numpy pandas matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Configuration & imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

RNG = np.random.default_rng(42)

N_ASSETS = 30
N_DAYS   = 45
HOURS    = N_DAYS * 24
GAMMA    = 0.004     # temp coefficient of power, ~ -0.4 %/°C for c-Si
NOCT     = 45        # nominal operating cell temperature (°C)
TODAY_WINDOW_H  = 48 # the "last 2 days" the morning triage looks at
MATERIALITY_MWH = 5.0
PERSIST_WIN = 4      # window of recent daytime hours for the persistence test
PERSIST_MIN = 3      # require >= 3 of last 4 daytime hours to dip (kills noise)
DAY_IRR  = 150       # only learn/judge PR when there is real sun
REGIONS  = ["Southwest", "Southeast", "Midwest"]
OEMS     = ["OEM-Alpha", "OEM-Bravo", "OEM-Charlie"]

## 2. Synthetic data — asset metadata
One row per asset: capacity, region, OEM, the latent *healthy* PR each asset operates around, and its contract (≈60% PPA / fixed price, rest merchant / volatile).

In [3]:
asset_ids = [f"SOLAR_{i:03d}" for i in range(1, N_ASSETS + 1)]
meta = pd.DataFrame({
    "asset_id": asset_ids,
    "capacity_mw": RNG.choice([5, 10, 20, 50, 80, 120, 200], N_ASSETS),
    "region": RNG.choice(REGIONS, N_ASSETS),
    "oem": RNG.choice(OEMS, N_ASSETS),
    "true_pr": RNG.uniform(0.78, 0.86, N_ASSETS),
    "contract_type": RNG.choice(["PPA", "Merchant"], N_ASSETS, p=[0.6, 0.4]),
    "ppa_rate": RNG.uniform(40, 90, N_ASSETS),   # USD/MWh — US utility-scale solar PPA range
}).set_index("asset_id")
meta.head(100)

,capacity_mw,region,oem,true_pr,contract_type,ppa_rate
asset_id,,,,,,
SOLAR_001,5,Southeast,OEM-Charlie,0.844381,PPA,75.844509
SOLAR_002,120,Southwest,OEM-Charlie,0.810998,PPA,62.468075
SOLAR_003,80,Southwest,OEM-Bravo,0.803066,Merchant,53.612078
SOLAR_004,50,Southeast,OEM-Charlie,0.834600,PPA,44.819548
SOLAR_005,50,Midwest,OEM-Bravo,0.791180,Merchant,85.130120
SOLAR_006,200,Southwest,OEM-Alpha,0.795993,Merchant,62.788814
SOLAR_007,5,Midwest,OEM-Charlie,0.780589,PPA,50.118168
SOLAR_008,80,Midwest,OEM-Bravo,0.842954,Merchant,55.297831
SOLAR_009,10,Southwest,OEM-Alpha,0.833188,PPA,68.960978


## 3. Shared weather drivers
Clear-sky irradiance is a half-sine over daylight hours with a small seasonal drift; a daily cloud factor scales it. Cell temperature follows the Sandia NOCT approximation.

In [4]:
idx  = pd.date_range("2025-04-01", periods=HOURS, freq="h")
hour = idx.hour.values
day  = (idx - idx[0]).days

solar_elev = np.clip(np.sin((hour - 6) / 12 * np.pi), 0, None)
seasonal   = 1 + 0.05 * np.sin(2 * np.pi * day / 365)
clear_sky  = 1000 * solar_elev * seasonal
cloud      = RNG.uniform(0.55, 1.0, N_DAYS)[day]
base_irr   = clear_sky * cloud
ambient_t  = 22 + 8 * solar_elev + 3 * np.sin(2 * np.pi * day / 365)

def cell_temp(irr, amb):
    return amb + (irr / 800.0) * (NOCT - 20)

## 4. Per-asset telemetry (healthy world)
Generation = potential × true_PR, with sensor noise on irradiance and diffuse noise on output. Faults are injected afterwards so we keep ground truth.

In [5]:
rows = []
for aid in asset_ids:
    m   = meta.loc[aid]
    irr = np.clip(base_irr * RNG.uniform(0.97, 1.03, HOURS), 0, None)
    amb = ambient_t + RNG.normal(0, 1.0, HOURS)
    ct  = cell_temp(irr, amb)
    potential = np.clip(m.capacity_mw * (irr / 1000.0) * (1 - GAMMA * (ct - 25)), 0, None)
    gen = potential * m.true_pr
    rows.append(pd.DataFrame({
        "timestamp": idx, "asset_id": aid,
        "irradiance": irr, "ambient_temp": amb, "cell_temp": ct,
        "potential_mwh": potential, "generation_mwh": gen,
        "availability_pct": RNG.uniform(0.985, 1.0) * 100,
        "inverter_status": "ok", "curtailment_flag": 0,
        "price": np.where(m.contract_type == "PPA", m.ppa_rate,
                          RNG.uniform(20, 150, HOURS)),  # USD/MWh — US wholesale spot range
    }))
data = pd.concat(rows, ignore_index=True)
data["region"] = data.asset_id.map(meta["region"])
data["generation_mwh"] *= RNG.normal(1.0, 0.03, len(data)).clip(0.85, 1.15)
data["generation_mwh"] = data["generation_mwh"].clip(lower=0)
print(data.shape)

(32400, 12)


In [6]:
data['asset_id'].value_counts()

asset_id
SOLAR_001    1080
SOLAR_002    1080
SOLAR_003    1080
SOLAR_004    1080
SOLAR_005    1080
SOLAR_006    1080
SOLAR_007    1080
SOLAR_008    1080
SOLAR_009    1080
SOLAR_010    1080
SOLAR_011    1080
SOLAR_012    1080
SOLAR_013    1080
SOLAR_014    1080
SOLAR_015    1080
SOLAR_016    1080
SOLAR_017    1080
SOLAR_018    1080
SOLAR_019    1080
SOLAR_020    1080
SOLAR_021    1080
SOLAR_022    1080
SOLAR_023    1080
SOLAR_024    1080
SOLAR_025    1080
SOLAR_026    1080
SOLAR_027    1080
SOLAR_028    1080
SOLAR_029    1080
SOLAR_030    1080
Name: count, dtype: int64

In [7]:
data[data['asset_id']=='SOLAR_005']

,timestamp,asset_id,irradiance,ambient_temp,cell_temp,potential_mwh,generation_mwh,availability_pct,inverter_status,curtailment_flag,price,region
4320,2025-04-01 00:00:00,SOLAR_005,0.0,22.424295,22.424295,0.0,0.0,99.490091,ok,0,46.008199,Midwest
4321,2025-04-01 01:00:00,SOLAR_005,0.0,21.693770,21.693770,0.0,0.0,99.490091,ok,0,90.568571,Midwest
4322,2025-04-01 02:00:00,SOLAR_005,0.0,20.962537,20.962537,0.0,0.0,99.490091,ok,0,21.934407,Midwest
4323,2025-04-01 03:00:00,SOLAR_005,0.0,21.430107,21.430107,0.0,0.0,99.490091,ok,0,86.150481,Midwest
4324,2025-04-01 04:00:00,SOLAR_005,0.0,21.300161,21.300161,0.0,0.0,99.490091,ok,0,71.169103,Midwest
...,...,...,...,...,...,...,...,...,...,...,...,...
5395,2025-05-15 19:00:00,SOLAR_005,0.0,24.194672,24.194672,0.0,0.0,99.490091,ok,0,113.753934,Midwest
5396,2025-05-15 20:00:00,SOLAR_005,0.0,24.919593,24.919593,0.0,0.0,99.490091,ok,0,123.166723,Midwest
5397,2025-05-15 21:00:00,SOLAR_005,0.0,22.691475,22.691475,0.0,0.0,99.490091,ok,0,148.162716,Midwest
5398,2025-05-15 22:00:00,SOLAR_005,0.0,22.487744,22.487744,0.0,0.0,99.490091,ok,0,52.919079,Midwest


## 5. Inject ground-truth faults
Four assets get a known, distinct fault so we can validate that detection + attribution recover them:

| Asset | Fault | Signature |
|---|---|---|
| `SOLAR_005` | Inverter **outage** | abrupt PR drop, availability ↓, inverter=fault (last 2 days) |
| `SOLAR_012` | **Degradation** | slow PR decay over the whole window |
| `SOLAR_019` | Frozen **sensor** | irradiance stuck at a constant (last 3 days) |
| `SOLAR_023` | **Curtailment** | recent capping, flag set, availability normal |
| `SOLAR_027` | **Mixed** | curtailment (day 1) + partial outage (day 2) — makes diagnostic consistency < 1.0 |
| all `Southwest` | **Resource/Weather** | a regional afternoon dip the sensors under-captured — peers fall together |

In [8]:
fault_log = {}
def mask(aid, d0, d1):
    return ((data.asset_id == aid)
            & (data.timestamp >= idx[0] + pd.Timedelta(days=d0))
            & (data.timestamp <  idx[0] + pd.Timedelta(days=d1)))

# (A) Inverter outage — abrupt, recent
mo = mask("SOLAR_005", 43, 45) & (data.irradiance > 50)
data.loc[mo, "generation_mwh"] *= 0.45
data.loc[mo, "availability_pct"] = 52.0
data.loc[mo, "inverter_status"] = "fault"
fault_log["SOLAR_005"] = "Outage"

# (B) Gradual degradation — slow PR decay
md_ = data.asset_id == "SOLAR_012"
decay = 1 - 0.18 * (data.loc[md_, "timestamp"] - idx[0]).dt.days / N_DAYS
data.loc[md_, "generation_mwh"] *= decay.values
fault_log["SOLAR_012"] = "Degradation"

# (C) Frozen sensor — irradiance stuck
ms = mask("SOLAR_019", 42, 45)
data.loc[ms, "irradiance"] = 430.0
fault_log["SOLAR_019"] = "Sensor/Data issue"

# (D) Curtailment — capping with flag set
mc = mask("SOLAR_023", 43, 45) & (data.irradiance > 200)
data.loc[mc, "generation_mwh"] *= 0.40
data.loc[mc, "curtailment_flag"] = 1
fault_log["SOLAR_023"] = "Curtailment / Market"

# (E) Mixed — curtailment day 1, partial outage day 2 (same asset) -> confidence < 1
me1 = mask("SOLAR_027", 43, 44) & (data.irradiance > 200)
data.loc[me1, "generation_mwh"] *= 0.45
data.loc[me1, "curtailment_flag"] = 1
me2 = mask("SOLAR_027", 44, 45) & (data.irradiance > 50)
data.loc[me2, "generation_mwh"] *= 0.55
data.loc[me2, "availability_pct"] = 68.0
data.loc[me2, "inverter_status"] = "fault"
fault_log["SOLAR_027"] = "Mixed (curtailment + outage)"

# (F) Regional resource/weather — every Southwest asset dips one afternoon,
#     measured irradiance unchanged -> peers fall together -> attributed weather
reg_assets = meta.index[meta.region == "Southwest"]
mw = (data.asset_id.isin(reg_assets)
      & (data.timestamp >= idx[0] + pd.Timedelta(days=44, hours=11))
      & (data.timestamp <  idx[0] + pd.Timedelta(days=44, hours=17)))
data.loc[mw, "generation_mwh"] *= 0.62
fault_log["REGION:Southwest"] = "Resource/Weather (regional)"

# recompute potential with the (possibly corrupted) sensor reading
data["potential_mwh"] = (
    data["asset_id"].map(meta["capacity_mw"]) * (data["irradiance"] / 1000.0)
    * (1 - GAMMA * (data["cell_temp"] - 25))
).clip(lower=0)
fault_log

{'SOLAR_005': 'Outage',
 'SOLAR_012': 'Degradation',
 'SOLAR_019': 'Sensor/Data issue',
 'SOLAR_023': 'Curtailment / Market',
 'SOLAR_027': 'Mixed (curtailment + outage)',
 'REGION:Southwest': 'Resource/Weather (regional)'}

## 6. Baseline — learn `PR_baseline` from healthy history
**This is the leakage-critical step.** We learn the baseline PR only on hours that are: daylight, available, *not* curtailed, and within a plausible PR range — and only from the **first 30 days** (so a fault in the recent window can't teach the baseline to accept itself). Robust **median**, not mean.

In [9]:
data["pr_obs"] = np.where(data.potential_mwh > 1e-6,
                          data.generation_mwh / data.potential_mwh, np.nan)
healthy = (
    (data.irradiance > DAY_IRR) & (data.availability_pct > 90)
    & (data.curtailment_flag == 0) & (data.pr_obs.between(0.3, 1.1))
)
train = data[healthy & (data.timestamp < idx[0] + pd.Timedelta(days=30))]
pr_base = train.groupby("asset_id")["pr_obs"].median()
pr_lo   = train.groupby("asset_id")["pr_obs"].quantile(0.10)  # lower confidence band
data["pr_baseline"] = data.asset_id.map(pr_base)
data["pr_band_lo"]  = data.asset_id.map(pr_lo)
data["expected_mwh"] = data.potential_mwh * data.pr_baseline
data[["asset_id","pr_baseline","pr_band_lo"]].drop_duplicates().head()

,asset_id,pr_baseline,pr_band_lo
0,SOLAR_001,0.846710,0.805182
1080,SOLAR_002,0.809562,0.777937
2160,SOLAR_003,0.803870,0.773897
3240,SOLAR_004,0.829996,0.799490
4320,SOLAR_005,0.791467,0.764795


## 7. Anomaly detection — *sustained* shortfall below the healthy PR band
A raw dip (PR below the asset's own 10th-percentile band, with real sun) is noisy: a p10 band crosses ~10% of healthy hours by construction. We add a **persistence test** — a dip only counts if ≥ `PERSIST_MIN` of the last `PERSIST_WIN` daytime hours dip — because real faults span many hours while p10 noise is isolated. Physically impossible PR is a data-quality tell and flags immediately.

In [10]:
data = data.sort_values(["asset_id", "timestamp"]).reset_index(drop=True)
DAY = data.irradiance > DAY_IRR
data["raw_under"] = (
    DAY & (data.pr_obs < data.pr_band_lo) & (data.generation_mwh < data.expected_mwh)
).astype(int)

day_rows = data[DAY].copy()
day_rows["roll"] = (day_rows.groupby("asset_id")["raw_under"]
                    .transform(lambda s: s.rolling(PERSIST_WIN, min_periods=1).sum()))
day_rows["persist"] = (day_rows["roll"] >= PERSIST_MIN) & (day_rows["raw_under"] == 1)
data["persist_under"] = False
data.loc[day_rows.index, "persist_under"] = day_rows["persist"].values

impossible = (~data.pr_obs.between(0.0, 1.15)) & DAY      # physically impossible PR
data["underperf"] = data["persist_under"] | impossible

# peer-group signal: fraction of same-region assets ALSO below expected this hour.
# High fraction => shared (regional weather) cause, not an asset-specific fault.
data["below_exp"] = ((data.generation_mwh < data.expected_mwh * 0.90)
                     & (data.irradiance > DAY_IRR)).astype(int)
data["regional_dip_frac"] = (data.groupby(["region", "timestamp"])["below_exp"]
                             .transform("mean"))
print("flagged hours:", int(data.underperf.sum()))

flagged hours: 325


## 8. Cause attribution — ordered rules over auxiliary signals
Order matters: check the **data input first** (don't diagnose a plant when the sensor is the problem), then curtailment, then outage, then a **peer-group test** for regional weather (same-region peers dipping together), and finally escalate. Degradation is a *slow* signal, detected at asset level from the multi-week PR trend. Weather/resource-*normal* needs no rule — the baseline normalizes it away, so it simply never flags.

In [11]:
data = data.sort_values(["asset_id", "timestamp"])
data["irr_std6"] = (data.groupby("asset_id")["irradiance"]
                    .transform(lambda s: s.rolling(6, min_periods=3).std()))

def attribute(r):
    if not r.underperf:
        return "OK"
    if (r.pr_obs > 1.15) or (r.pr_obs < 0) or (r.irradiance > 300 and r.irr_std6 < 1.0):
        return "Sensor/Data issue"
    if r.curtailment_flag == 1:
        return "Curtailment / Market"
    if (r.availability_pct < 90) or (r.inverter_status == "fault"):
        return "Outage"
    if r.regional_dip_frac >= 0.5:          # peers fall together => weather, not a fault
        return "Resource/Weather"
    return "Unexplained"

data["cause"] = data.apply(attribute, axis=1)

# degradation: asset-level PR trend
for aid, g in data[healthy.values | (data.cause == "Unexplained")].groupby("asset_id"):
    gg = g.dropna(subset=["pr_obs"])
    if len(gg) < 50:
        continue
    t = (gg.timestamp - gg.timestamp.min()).dt.total_seconds().values
    slope = np.polyfit(t, gg.pr_obs.values, 1)[0] * 3600 * 24 * N_DAYS
    if slope < -0.08:
        m_un = (data.asset_id == aid) & (data.cause == "Unexplained")
        data.loc[m_un, "cause"] = "Degradation"

data.cause.value_counts()

cause
OK                      32075
Degradation               134
Sensor/Data issue          55
Resource/Weather           41
Unexplained                35
Outage                     31
Curtailment / Market       29
Name: count, dtype: int64

## 9. Financial impact + daily triage (last 48h)
Lost MWh × the **contract-relevant price** (fixed PPA vs. volatile merchant). Sensor/data issues are surfaced as a *data-quality* flag with unknown \$ impact — not buried at \$0. **Resource/Weather** events are non-actionable (you can't fix the weather), so they are pulled out of the action list and reported as a one-line regional summary. The score below is **diagnostic consistency** — the share of an asset's flagged hours agreeing with the dominant cause — *not* a calibrated probability.

In [12]:
from IPython.display import display

NEXT_STEP = {
    "Outage": "Dispatch field tech; pull inverter/string fault logs",
    "Curtailment / Market": "Confirm dispatch instruction; split economic vs. recoverable curtailment",
    "Degradation": "Schedule soiling/module inspection; review long-term PR trend",
    "Sensor/Data issue": "Validate POA sensor calibration; check telemetry pipeline freshness",
    "Resource/Weather": "No action - regional resource event; confirm vs. forecast, no dispatch",
    "Unexplained": "Manual review by performance engineer",
}
recent = data[data.timestamp >= idx[-1] - pd.Timedelta(hours=TODAY_WINDOW_H)].copy()
recent["lost_mwh"] = (recent.expected_mwh - recent.generation_mwh).clip(lower=0)
flagged = recent[recent.cause != "OK"].copy()
flagged.loc[flagged.cause == "Sensor/Data issue", "lost_mwh"] = 0.0
flagged["lost_rev"] = flagged.lost_mwh * flagged.price

agg = flagged.groupby("asset_id").agg(
    lost_mwh_48h=("lost_mwh", "sum"),
    est_lost_revenue=("lost_rev", "sum"),
    flagged_hours=("cause", "size"),
    probable_cause=("cause", lambda s: s.value_counts().index[0]),
).reset_index()
# diagnostic CONSISTENCY (share of flagged hours on the dominant cause) - NOT a
# calibrated probability; calibration would come from operator-confirmed labels.
cons = flagged.groupby("asset_id")["cause"].apply(lambda s: s.value_counts().iloc[0] / len(s))
agg["diagnostic_consistency"] = agg.asset_id.map(cons).round(2)
second = flagged.groupby("asset_id")["cause"].apply(
    lambda s: s.value_counts().index[1] if s.nunique() > 1 else None)
agg["secondary_cause"] = agg.asset_id.map(second)
agg = agg.join(meta[["region", "capacity_mw", "contract_type"]], on="asset_id")
agg["next_step"] = agg.probable_cause.map(NEXT_STEP)
lc = (agg.diagnostic_consistency < 0.7) & agg.secondary_cause.notna()
agg.loc[lc, "next_step"] = ("[Mixed: also " + agg.loc[lc, "secondary_cause"]
                            + "] " + agg.loc[lc, "next_step"])

# Resource/Weather is shared & non-actionable: pull out, summarize by region
resource = agg[agg.probable_cause == "Resource/Weather"].copy()
agg = agg[agg.probable_cause != "Resource/Weather"].copy()

keep = (agg.lost_mwh_48h >= MATERIALITY_MWH) | (agg.probable_cause == "Sensor/Data issue")
agg = agg[keep].copy()
agg["flag_type"] = np.where(agg.probable_cause == "Sensor/Data issue", "Data quality", "Performance")
agg["est_lost_revenue"] = np.where(agg.flag_type == "Data quality", np.nan, agg.est_lost_revenue)
agg = agg.sort_values(["flag_type", "est_lost_revenue"], ascending=[True, False]).reset_index(drop=True)

triage = agg[["asset_id","region","capacity_mw","contract_type","flag_type",
              "probable_cause","diagnostic_consistency","lost_mwh_48h","est_lost_revenue",
              "flagged_hours","next_step"]].copy()
triage["lost_mwh_48h"]     = triage.lost_mwh_48h.round(1)
triage["est_lost_revenue"] = triage.est_lost_revenue.round(0)

res_summary = (resource.groupby("region")
               .agg(assets=("capacity_mw","size"), lost_mwh_48h=("lost_mwh_48h","sum"))
               .round(1).reset_index())

print("Non-actionable regional resource events:")
display(res_summary if len(res_summary) else "  none")
print("\nPerformance triage (last 48 h):")
triage

Non-actionable regional resource events:


,region,assets,lost_mwh_48h
0,Southwest,9,734.7



Performance triage (last 48 h):


,asset_id,region,capacity_mw,contract_type,flag_type,probable_cause,diagnostic_consistency,lost_mwh_48h,est_lost_revenue,flagged_hours,next_step
0,SOLAR_019,Midwest,120,PPA,Data quality,Sensor/Data issue,1.00,0.0,NaN,41,Validate POA sensor calibration; check telemet...
1,SOLAR_005,Midwest,50,Merchant,Performance,Outage,1.00,262.6,22465.0,20,Dispatch field tech; pull inverter/string faul...
2,SOLAR_012,Midwest,200,Merchant,Performance,Degradation,1.00,239.2,18685.0,22,Schedule soiling/module inspection; review lon...
3,SOLAR_027,Southeast,20,Merchant,Performance,Outage,0.55,97.2,7500.0,20,[Mixed: also Curtailment / Market] Dispatch fi...
4,SOLAR_023,Southeast,10,PPA,Performance,Curtailment / Market,1.00,58.8,2775.0,20,Confirm dispatch instruction; split economic v...


## 10. Validation — did we recover the injected faults?
The six ground-truth scenarios should be recovered: four single-cause faults at consistency 1.0, the mixed asset on one of its two causes at **consistency < 1.0**, and the regional weather event collapsed into a non-actionable resource summary. The false-positive ("Unexplained") tail should be ~0.

In [ ]:
print("Injected:", fault_log)
print()
for aid, truth in fault_log.items():
    if aid.startswith("REGION:"):
        reg = aid.split(":")[1]
        n = int((resource.region == reg).sum()) if len(resource) else 0
        print(f"[OK  ] {aid}: {n} {reg} assets -> 'Resource/Weather' (non-actionable)  truth={truth!r}")
        continue
    row = triage[triage.asset_id == aid]
    if len(row) == 0:
        print(f"[MISS] {aid}: not in triage"); continue
    r = row.iloc[0]
    tag = "OK  " if (r.probable_cause in truth or truth in r.probable_cause
                     or ("Mixed" in truth and r.probable_cause in ("Outage","Curtailment / Market"))) else "MISS"
    print(f"[{tag}] {aid}: predicted={r.probable_cause!r}  consistency={r.diagnostic_consistency}  truth={truth!r}")

n_unexpl = int((triage.probable_cause == "Unexplained").sum())
print(f"\nFalse-positive tail (Unexplained rows in triage): {n_unexpl}")

## 11. Figures

In [ ]:
import matplotlib.dates as mdates

BG = "#F7F8FA"

CAUSE_PALETTE = {
    "Outage":               "#D94F3D",
    "Degradation":          "#5B4FCF",
    "Sensor/Data issue":    "#2F86C8",
    "Curtailment / Market": "#E8922A",
    "Mixed (curtail+outage)": "#9B59B6",
}

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#E0E0E0",
    "grid.linewidth":    0.6,
    "grid.alpha":        1.0,
    "axes.axisbelow":    True,
    "legend.labelcolor": "#333333",
})

examples = [
    ("SOLAR_005", "Outage"),
    ("SOLAR_012", "Degradation"),
    ("SOLAR_019", "Sensor/Data issue"),
    ("SOLAR_023", "Curtailment / Market"),
    ("SOLAR_027", "Mixed (curtail+outage)"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 7), facecolor=BG)
fig.patch.set_facecolor(BG)
axl = axes.ravel()

x_end       = data.timestamp.max()
x_start     = x_end - pd.Timedelta(days=10)
fault_onset = x_end - pd.Timedelta(hours=TODAY_WINDOW_H)

for ax, (aid, label) in zip(axl, examples):
    g  = data[(data.asset_id == aid) & (data.timestamp >= x_start)].copy()
    fl = g[g.cause != "OK"]
    cc = CAUSE_PALETTE.get(label, "#888888")

    ax.axvspan(fault_onset, x_end, color=cc, alpha=0.07, zorder=0)
    ax.axvline(fault_onset, color=cc, lw=0.9, ls="--", alpha=0.55, zorder=1)

    ax.fill_between(g.timestamp, g.expected_mwh, color="#2E8B7A", alpha=0.10, zorder=2)
    ax.plot(g.timestamp, g.expected_mwh, color="#2E8B7A", lw=1.6, label="Expected", zorder=3)
    ax.plot(g.timestamp, g.generation_mwh, color="#3A5BA0", lw=1.1, label="Actual", zorder=4)

    if len(fl):
        ax.scatter(fl.timestamp, fl.generation_mwh, color=cc, s=20, zorder=6,
                   edgecolors="white", linewidths=0.5, label="Flagged")

    ax.set_facecolor(BG)
    ax.spines["left"].set_color("#CCCCCC")
    ax.spines["bottom"].set_color("#CCCCCC")

    ax.text(0.99, 0.97, label, transform=ax.transAxes,
            fontsize=7, ha="right", va="top", color="white", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.35", facecolor=cc, edgecolor="none", alpha=0.92))

    ax.set_title(aid, fontsize=9.5, fontweight="bold", color="#1A1A2E", pad=6, loc="left")
    ax.set_ylabel("MWh", fontsize=8, color="#777777")
    ax.tick_params(axis="x", labelrotation=35, labelsize=6.5, colors="#666666")
    ax.tick_params(axis="y", labelsize=7, colors="#666666")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))

    leg_handles = [
        plt.Line2D([0], [0], color="#2E8B7A", lw=1.5, label="Expected"),
        plt.Line2D([0], [0], color="#3A5BA0", lw=1.1, label="Actual"),
        plt.Line2D([0], [0], color=cc, lw=0, marker="o", markersize=5,
                   markeredgecolor="white", markeredgewidth=0.4, label="Flagged"),
    ]
    ax.legend(handles=leg_handles, fontsize=6.5, loc="upper left",
              frameon=True, framealpha=0.88, edgecolor="#DDDDDD",
              facecolor="white", labelcolor="#333333", labelspacing=0.25,
              handlelength=1.2, handletextpad=0.4)

axl[-1].axis("off")
axl[-1].set_facecolor(BG)

# reserve top space so title and subtitle don't overlap the subplots
fig.tight_layout(rect=[0, 0, 1, 0.89])

fig.suptitle("Expected (PR Baseline) vs. Actual Generation — Last 10 Days",
             fontsize=13, fontweight="bold", color="#1A1A2E", y=0.97)
fig.text(0.5, 0.92,
         "Shaded region = triage window (last 48 h)  |"
         "  dashed line = fault onset  |  dots = flagged hours",
         ha="center", fontsize=8, color="#888888")

plt.savefig("fault_timeseries.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()

In [ ]:
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

CAUSE_COLORS = {
    "Outage":               "#D94F3D",
    "Curtailment / Market": "#E8922A",
    "Degradation":          "#5B4FCF",
    "Sensor/Data issue":    "#2F86C8",
    "Unexplained":          "#8A8A8A",
}
CAUSE_TAGS = {
    "Outage":               "[OUT]",
    "Curtailment / Market": "[CURT]",
    "Degradation":          "[DEG]",
    "Sensor/Data issue":    "[SENS]",
    "Unexplained":          "[?]",
}

plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         False,
})

top = triage[triage.flag_type == "Performance"].head(10).iloc[::-1].copy()
top["label_y"] = [
    f"{row.asset_id}\n{row.region}  ·  {int(row.capacity_mw)} MW  ·  {row.contract_type}"
    for _, row in top.iterrows()
]

# ── figure ────────────────────────────────────────────────────────────────────
fig2 = plt.figure(figsize=(14, 5.5), facecolor="#F7F8FA")
gs   = GridSpec(1, 2, figure=fig2, width_ratios=[3, 1], wspace=0.04)
ax2  = fig2.add_subplot(gs[0])
axk  = fig2.add_subplot(gs[1])
ax2.set_facecolor("#F7F8FA")
axk.set_facecolor("#F7F8FA")

# ── zebra rows ────────────────────────────────────────────────────────────────
for i in range(len(top)):
    ax2.axhspan(i - 0.45, i + 0.45,
                color="#EBEBEB" if i % 2 == 0 else "#F7F8FA",
                zorder=0, linewidth=0)

# ── bars ──────────────────────────────────────────────────────────────────────
colors = [CAUSE_COLORS.get(c, "#8A8A8A") for c in top.probable_cause]
bars   = ax2.barh(
    top.label_y, top.est_lost_revenue,
    color=colors, height=0.55, edgecolor="white", linewidth=0.8, zorder=3,
)

# ── annotations (no conf score — too granular for presentation) ───────────────
max_rev = top.est_lost_revenue.max()
for bar, (_, row) in zip(bars, top.iterrows()):
    w   = bar.get_width()
    tag = CAUSE_TAGS.get(row.probable_cause, "")
    ax2.text(
        w + max_rev * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"${w:,.0f}   ({row.lost_mwh_48h:.0f} MWh)   {tag}",
        va="center", ha="left", fontsize=8.5,
        color="#333333", fontweight="bold",
    )

# ── axes cosmetics ────────────────────────────────────────────────────────────
ax2.set_xlim(0, max_rev * 1.55)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"${x/1e3:,.0f}k" if x >= 1000 else f"${x:,.0f}"))
ax2.tick_params(axis="x", labelsize=8.5, colors="#555555")
ax2.tick_params(axis="y", labelsize=8.5, colors="#222222", pad=6)
ax2.spines["left"].set_color("#CCCCCC")
ax2.spines["bottom"].set_color("#CCCCCC")
ax2.set_xlabel("Estimated lost revenue — last 48 h  (USD)",
               fontsize=9.5, color="#555555", labelpad=8)

ax2.text(0.0, 1.12, "Morning Triage · Performance Issues",
         transform=ax2.transAxes, fontsize=14, fontweight="bold",
         color="#1A1A2E", va="bottom")
ax2.text(0.0, 1.05, "Ranked by estimated revenue impact · last 48 h · solar fleet",
         transform=ax2.transAxes, fontsize=9, color="#6B6B6B", va="bottom")

legend_causes  = [c for c in CAUSE_COLORS if c in top.probable_cause.values]
legend_handles = [
    Patch(facecolor=CAUSE_COLORS[c], edgecolor="white", linewidth=0.5, label=c)
    for c in legend_causes
]
ax2.legend(handles=legend_handles, fontsize=8, loc="lower right",
           frameon=True, framealpha=0.9, edgecolor="#DDDDDD",
           facecolor="white", labelcolor="#333333", labelspacing=0.5)

# ── KPI side panel ────────────────────────────────────────────────────────────
axk.axis("off")
axk.set_xlim(0, 1)
axk.set_ylim(0, 1)

total_rev  = top.est_lost_revenue.sum()
total_mwh  = top.lost_mwh_48h.sum()
n_assets   = len(top)
n_dispatch = int((top.probable_cause == "Outage").sum())
reg_mwh    = res_summary.lost_mwh_48h.sum() if len(res_summary) else 0.0

kpis = [
    ("Total at risk",      f"${total_rev/1e3:,.1f}k",  "#D94F3D"),
    ("Lost generation",    f"{total_mwh:,.0f} MWh",     "#5B4FCF"),
    ("Assets flagged",     str(n_assets),                "#333333"),
    ("Dispatch required",  str(n_dispatch),              "#E8922A"),
    ("Regional (weather)", f"{reg_mwh:,.0f} MWh",       "#2F86C8"),
]

axk.text(0.5, 0.97, "Key Metrics", ha="center", va="top",
         fontsize=10, fontweight="bold", color="#1A1A2E")
axk.plot([0.05, 0.95], [0.92, 0.92], color="#CCCCCC", linewidth=0.8,
         transform=axk.transAxes, clip_on=False)

for i, (lbl, val, col) in enumerate(kpis):
    y = 0.84 - i * 0.165
    axk.add_patch(plt.Rectangle(
        (0.05, y - 0.06), 0.90, 0.12,
        transform=axk.transAxes, clip_on=False,
        facecolor="white", edgecolor="#DDDDDD", linewidth=0.7, zorder=2,
    ))
    axk.text(0.50, y + 0.015, val, ha="center", va="center",
             fontsize=12, fontweight="bold", color=col,
             transform=axk.transAxes, zorder=3)
    axk.text(0.50, y - 0.035, lbl, ha="center", va="center",
             fontsize=7.5, color="#7A7A7A",
             transform=axk.transAxes, zorder=3)

# ── regional weather footnote ─────────────────────────────────────────────────
if len(res_summary):
    reg_str = ", ".join(
        f"{r.region} ({r.assets} assets, {r.lost_mwh_48h:.0f} MWh)"
        for _, r in res_summary.iterrows()
    )
    ax2.text(0.0, -0.14,
             f"Non-actionable regional resource/weather event — {reg_str}",
             transform=ax2.transAxes, fontsize=8, color="#888888", style="italic")

fig2.subplots_adjust(left=0.14, right=0.98, top=0.84, bottom=0.18)
plt.savefig("triage_chart.png", dpi=150, bbox_inches="tight",
            facecolor=fig2.get_facecolor())
plt.show()